### Things I need for this project:

1. SpaceDevs API
2. Some sort of weather API
3. Fast API/Flask

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import requests
from datetime import datetime
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [11]:
rocket_launch_api = "https://ll.thespacedevs.com/2.3.0/launches/"
open_mateo_weatherapi = "https://archive-api.open-meteo.com/v1/archive"

In [15]:
def get_weather_data(latitude, longitude, timestamp):
    #extracting date & hour
    #consider the following timestamp: "2024-02-10T14:00:00"
    #We need to split this into two lists which can be done by spliting at T
    #So then we have: []"2024-02-10", "14:00:00"] Now its very easy to access the date and hour
    try:
        date = timestamp.split("T")[0]
        hour = int(timestamp.split("T")[1].split(":")[0])

        parameters = {
            "latitude": latitude,
            "longitude": longitude,
            "start_date": date,
            "end_date": date,
            "hourly": "temperature_2m, precipitation, windspeed_10m, cloudcover", #I'm only focused on these specific weather features for this model
            "timezone": "UTC"
        }

    #making the request
        request = requests.get(open_mateo_weatherapi, params=parameters)

        weather_data = request.json()["hourly"] #now each of the 4 lists I made is going to contain 24 values

        return { #returning the weather at exactly that hour
            "temperature" : weather_data["temperature_2m"][hour],
            "precipitation" : weather_data["precipitaion"][hour],
            "wind_speed": weather_data["windspeed_10m"][hour],
            "cloud_cover": weather_data["cloudcover"][hour]
        }
    except Exception as e:
        return {"temp": None, "precip": None, "wind_speed": None, "cloud_cover": None}


In [ ]:
def get_launch_date():
    #I only want SPACEX rocket data
    
    limit = 150 #I want 150 launches
    all_launches = []

    parameters = {
        "limit" : 100,
        "lsp__name" : "SpaceX",
        "mode": "detailed"
    }

    # a problem with this website is that there are multiple pages all the way through 781
    # so we need a loop to go through all this
    url = rocket_launch_api

    while url and len(all_launches) < limit:
        try:
            request = requests.get(url, params=parameters)
            rocket_data = request.json()

            all_launches.extend(rocket_data["results"])
        
            print(f"Fetched {len(all_launches)} launches so far...")

            url = rocket_data.get("next")
            parameters = {}

            time.sleep(1)
        
        except requests.exceptions.RequestException as e:
            print("Error getting Rocket Data")
            break
        

